In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf

In [3]:
spark = SparkSession.builder.appName("case_study").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 04:27:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/15 04:27:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
df_customer = spark.read.csv("data/customers.csv", header = True, inferSchema = True)
df_customer.take(5)

[Row(customer_id=1, customer_name='Customer_1', city='Columbus', state='OH', registration_date=datetime.date(2023, 10, 17), customer_segment='VIP'),
 Row(customer_id=2, customer_name='Customer_2', city='Miami', state='CA', registration_date=datetime.date(2022, 4, 25), customer_segment='Premium'),
 Row(customer_id=3, customer_name='Customer_3', city='Atlanta', state='FL', registration_date=datetime.date(2022, 1, 26), customer_segment='Premium'),
 Row(customer_id=4, customer_name='Customer_4', city='Chicago', state='OH', registration_date=datetime.date(2022, 10, 9), customer_segment='Standard'),
 Row(customer_id=5, customer_name='Customer_5', city='Columbus', state='IL', registration_date=datetime.date(2022, 9, 8), customer_segment='Premium')]

In [6]:
df_item = spark.read.csv("data/order_items.csv", header = True, inferSchema = True)
df_item.take(5)

[Row(order_item_id=1, order_id=227444, product_id=28849, quantity=5, selling_price=727.98),
 Row(order_item_id=2, order_id=32708, product_id=25471, quantity=2, selling_price=203.02),
 Row(order_item_id=3, order_id=426242, product_id=2818, quantity=5, selling_price=1061.81),
 Row(order_item_id=4, order_id=236965, product_id=35931, quantity=5, selling_price=1005.49),
 Row(order_item_id=5, order_id=289552, product_id=48310, quantity=4, selling_price=540.78)]

In [7]:
df_order = spark.read.csv("data/orders.csv", header = True, inferSchema = True)
df_order.take(5)

[Row(order_id=1, customer_id=54630, order_date=datetime.date(2024, 1, 25), payment_mode='Credit Card', order_status='Delivered'),
 Row(order_id=2, customer_id=22415, order_date=datetime.date(2024, 7, 8), payment_mode='Credit Card', order_status='Delivered'),
 Row(order_id=3, customer_id=20909, order_date=datetime.date(2024, 1, 23), payment_mode='UPI', order_status='Delivered'),
 Row(order_id=4, customer_id=98027, order_date=datetime.date(2024, 3, 2), payment_mode='Credit Card', order_status='Cancelled'),
 Row(order_id=5, customer_id=31332, order_date=datetime.date(2024, 12, 17), payment_mode='UPI', order_status='Cancelled')]

In [8]:
df_product = spark.read.csv("data/products.csv", header = True, inferSchema = True)
df_product.take(5)

[Row(product_id=1, product_name='Product_1', category='Home & Kitchen', brand='Brand_A', unit_cost=509.39),
 Row(product_id=2, product_name='Product_2', category='Electronics', brand='Brand_A', unit_cost=332.22),
 Row(product_id=3, product_name='Product_3', category='Sports', brand='Brand_D', unit_cost=68.58),
 Row(product_id=4, product_name='Product_4', category='Clothing', brand='Brand_A', unit_cost=729.19),
 Row(product_id=5, product_name='Product_5', category='Home & Kitchen', brand='Brand_D', unit_cost=326.77)]

In [9]:
df_return = spark.read.csv("data/returns.csv", header = True, inferSchema = True)
df_return.take(5)

[Row(return_id=1, order_id=391349, return_date=datetime.date(2024, 8, 6), return_reason='Changed Mind'),
 Row(return_id=2, order_id=456171, return_date=datetime.date(2024, 6, 28), return_reason='Arrived Damaged'),
 Row(return_id=3, order_id=977675, return_date=datetime.date(2024, 9, 26), return_reason='Size Issue'),
 Row(return_id=4, order_id=80452, return_date=datetime.date(2024, 7, 30), return_reason='Arrived Damaged'),
 Row(return_id=5, order_id=10920, return_date=datetime.date(2024, 6, 15), return_reason='Size Issue')]

In [10]:
# 1. Total number of customers, products, orders, and returned orders

total_customers = df_customer.count()
total_products = df_product.count()
total_orders = df_order.count()
total_returned_orders = df_return.count()

print("Total Customers:", total_customers)
print("Total Products:", total_products)
print("Total Orders:", total_orders)
print("Total Returned Orders:", total_returned_orders)

Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


In [12]:
with open("output/q1_output.txt", "w") as f:
    f.write(f"Total Customers: {total_customers}\n")
    f.write(f"Total Products: {total_products}\n")
    f.write(f"Total Orders: {total_orders}\n")
    f.write(f"Total Returned Orders: {total_returned_orders}\n")

print("Question 1 output saved!")

Question 1 output saved!


In [11]:
# 2. Total sales amount generated by each product category

df_item.createOrReplaceTempView("order_items")
df_product.createOrReplaceTempView("products")

category_sales = spark.sql("""
    SELECT 
        p.category,
        SUM(oi.quantity * oi.selling_price) AS total_sales
    FROM order_items oi
    JOIN products p
    ON oi.product_id = p.product_id
    GROUP BY p.category
""")
category_sales.show()

[Stage 30:=============================>                            (1 + 1) / 2]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [13]:
category_sales.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q2_output")

In [14]:
# 3. Top 10 customers based on total purchase amount

df_customer.createOrReplaceTempView("customers")
df_order.createOrReplaceTempView("orders")
df_item.createOrReplaceTempView("order_items")

top_customers = spark.sql("""
    SELECT 
        c.customer_id,
        c.customer_name,
        SUM(oi.quantity * oi.selling_price) AS total_purchase
    FROM customers c JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_id, c.customer_name
    ORDER BY total_purchase DESC
    LIMIT 10
""")

top_customers.show()

[Stage 42:>                                                         (0 + 2) / 2]

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|169060.39999999997|
|      23289|Customer_23289|161573.80000000002|
|      52275|Customer_52275|153364.78999999998|
|      61218|Customer_61218|         153067.55|
|      52034|Customer_52034|         152680.05|
|      40442|Customer_40442|         151037.32|
|      60528|Customer_60528|         148691.95|
|      84830|Customer_84830|         148363.84|
|      82593|Customer_82593|148281.03999999998|
+-----------+--------------+------------------+



In [15]:
top_customers.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q3_output")
print("Question 3 output saved!")

[Stage 52:>                                                         (0 + 2) / 2]

Question 3 output saved!


In [16]:
# 4. Monthly sales trends for the last available year

monthly_sales = spark.sql("""
    SELECT 
        YEAR(o.order_date) AS year,
        MONTH(o.order_date) AS month,
        SUM(oi.quantity * oi.selling_price) AS total_sales
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    WHERE YEAR(o.order_date) = (
        SELECT MAX(YEAR(order_date))
        FROM orders
    )
    GROUP BY YEAR(o.order_date), MONTH(o.order_date)
    ORDER BY month
""")

monthly_sales.show()

[Stage 65:=============================>                            (1 + 1) / 2]

+----+-----+--------------------+
|year|month|         total_sales|
+----+-----+--------------------+
|2024|    1| 4.445777757600032E8|
|2024|    2| 4.153661441999996E8|
|2024|    3| 4.436282454099993E8|
|2024|    4|4.2782097433999574E8|
|2024|    5|4.4481061894999886E8|
|2024|    6| 4.317051540600023E8|
|2024|    7| 4.436705191199994E8|
|2024|    8|4.4109517702000153E8|
|2024|    9| 4.310715260799986E8|
|2024|   10|4.4136378931000113E8|
|2024|   11| 4.336233640400014E8|
|2024|   12| 4.427129083499967E8|
+----+-----+--------------------+



In [17]:
monthly_sales.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q4_output")
print("Question 4 output saved!")

Question 4 output saved!


In [18]:
spark.stop()